In [ ]:
#Example 1: Basic Usage
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CollectExample").getOrCreate()

# Sample DataFrame
data = [("Alice", 25, 3000), ("Bob", 30, 4000), ("Charlie", 28, 5000)]
df = spark.createDataFrame(data, ["name", "age", "salary"])

df.show()

# Collect all rows
result = df.collect()

print(result)



+-------+---+------+
|   name|age|salary|
+-------+---+------+
|  Alice| 25|  3000|
|    Bob| 30|  4000|
|Charlie| 28|  5000|
+-------+---+------+

[Row(name='Alice', age=25, salary=3000), Row(name='Bob', age=30, salary=4000), Row(name='Charlie', age=28, salary=5000)]


In [ ]:
#Example 2: Iterating Over Collected Data
rows = df.collect()

for row in rows:
    print(row["name"], row["age"], row["salary"])

Alice 25 3000
Bob 30 4000
Charlie 28 5000


In [ ]:
#Example 3: Convert to Pandas
#Often, after collecting, people convert to Pandas for local analysis:

pandas_df = df.toPandas()
print(pandas_df)

      name  age  salary
0    Alice   25    3000
1      Bob   30    4000
2  Charlie   28    5000


In [ ]:
#Equivalent of apply() on RDDs
#If you want Pandas-style apply() for row-wise operations, you can use map() on RDDs:

# Convert DataFrame to RDD
rdd = df.rdd

# Apply transformation (similar to row-wise apply)
rdd_applied = rdd.map(lambda row: (row["name"], row["age"] + 2))
print(rdd_applied.collect())

[('Alice', 27), ('Bob', 32), ('Charlie', 30)]


In [ ]:
#Example 4: Collect with Filtering

result = df.filter(df["age"] > 26).collect()
for row in result:
    print(row.name, row.salary)


Bob 4000
Charlie 5000


In [ ]:
from pyspark.sql.functions import col,upper,when
spark = SparkSession.builder.appName("Transform").getOrCreate()
data=[("Anni",25,30000),("Bob",28,40000),("Aman",28,60000)]
df=spark.createDataFrame(data,["name","age","salary"])
df.show()

+----+---+------+
|name|age|salary|
+----+---+------+
|Anni| 25| 30000|
| Bob| 28| 40000|
|Aman| 28| 60000|
+----+---+------+



In [ ]:
def add_bonus(dataframe):
    return dataframe.withColumn("bonus",col("salary")*0.1)

In [ ]:
df_transformed=df.transform(add_bonus)
df_transformed.show()

+----+---+------+------+
|name|age|salary| bonus|
+----+---+------+------+
|Anni| 25| 30000|3000.0|
| Bob| 28| 40000|4000.0|
|Aman| 28| 60000|6000.0|
+----+---+------+------+



In [ ]:
def uppercase_name(df):
  return df.withColumn("name",upper(col("name")))

In [ ]:
def categorize_salary(df):
  return df.withColumn("salary_category",when(col("salary")>55000,"High").when(col("salary")>35000,"Medium"))

In [ ]:
def pipeline(df):
  return(df.transform(add_bonus).transform(uppercase_name).transform(categorize_salary))

In [ ]:
df_pipeline=pipeline(df)
df_pipeline.show()

+----+---+------+------+---------------+
|name|age|salary| bonus|salary_category|
+----+---+------+------+---------------+
|ANNI| 25| 30000|3000.0|           NULL|
| BOB| 28| 40000|4000.0|         Medium|
|AMAN| 28| 60000|6000.0|           High|
+----+---+------+------+---------------+

